# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

This cell sets the system path and logger.

In [1]:
import sys
sys.path.append('../05_src/')

from utils.logger import get_logger
_logs = get_logger(__name__, log_dir='../../06_logs/')

_logs.info('This is a log message.')

2026-04-18 11:34:23,946, 40829820.py, 7, INFO, This is a log message.


In [2]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

This cell shows the selected artical "The GenAI Divide: State of AI in Business 2025" is loaded into a list "doc". Each element in "doc" is a LangChain Document object.

In [3]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "documents/ai_report_2025.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

This cell shows that this artical has 26 pages.

In [4]:
_logs.info(f"There are {len(docs)} pages in the artical.")

2026-04-18 11:34:27,288, 799000934.py, 1, INFO, There are 26 pages in the artical.


This cell joins the page content of all 26 pages into one string "document_text", i.e "flattened" the whole content of this article into one continuous string.

In [5]:
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

# print the number of characters in the article
_logs.info(f"This artical contains {len(document_text)} characters.")

2026-04-18 11:34:27,293, 880099285.py, 6, INFO, This artical contains 53851 characters.


This cell displays the beginning of the loaded article to check quality.

In [6]:
from IPython.display import display, Markdown

display(Markdown(document_text[:1000] + "... [Content Truncated]"))

pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI initiatives, structured 
interviews with representatives from 52 organizations, and survey responses from 
153 senior leaders collected across four major industry conferences. 
 Disclaimer: The views expressed in this report are solely those of the authors and 
reviewers and do not reflect the positions of any affiliated employers. 
 Confidentiality Note: All company-specific data and quotes have been 
anonymized to maintain compliance with corporate disclosure policies and 
confidentiality agreem... [Content Truncated]

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


This cell defines the Structured Output using a Pydantic BaseModel object.

In [7]:
from pydantic import BaseModel

class BookAnalysis(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

This cell initializes the OpenAI client with a custom configuration.

In [8]:
from openai import OpenAI
import os
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

This cell specifies the developer prompt.

In [9]:
system_prompt = f"""
          You are an AI scientist.
          You write the summary with a "Formal Academic Writing" tone.
"""

_logs.info(f"The developer prompt is: \n {system_prompt}")

2026-04-18 11:34:27,502, 1104809280.py, 6, INFO, The developer prompt is: 
 
          You are an AI scientist.
          You write the summary with a "Formal Academic Writing" tone.



This cell gives the user prompt.

In [10]:
prompt = f"""Given the following context from a book, do the following:
    
    1. Identify the book's author and title.
    2. Write a relevence statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    3. Write a concise and succinct summary no longer than 1000 tokens.
    4. Extract the tone used to produce the summary.
        
    The book is the following: 
    <book>
    {document_text}
    </book>
"""

_logs.info(f"The user prompt is: \n {prompt}")

2026-04-18 11:34:27,506, 3237998116.py, 14, INFO, The user prompt is: 
 Given the following context from a book, do the following:

    1. Identify the book's author and title.
    2. Write a relevence statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    3. Write a concise and succinct summary no longer than 1000 tokens.
    4. Extract the tone used to produce the summary.

    The book is the following: 
    <book>
    pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicl

This cell makes a call using response_format defined by a Pydantic BaseModel Object.

In [11]:
response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ],
    response_format=BookAnalysis,
)

This cell extracts the parsed object from response to "analysis".

In [12]:
analysis = response.choices[0].message.parsed

This cell prints out the outputs.

In [13]:
_logs.info(f"The output is: \n {analysis} \n")
_logs.info(f"The output Author is: \n {analysis.Author} \n")
_logs.info(f"The output Title is: \n {analysis.Title} \n")
_logs.info(f"The output Relevance is: \n {analysis.Relevance} \n")
_logs.info(f"The output Summary is: \n {analysis.Summary} \n")
_logs.info(f"The output Tone is: \n {analysis.Tone} \n")

2026-04-18 11:34:40,904, 3177562800.py, 1, INFO, The output is: 
 Author='Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari' Title='The GenAI Divide: State of AI in Business 2025' Relevance='This article is critically relevant for AI professionals as it explores the current landscape of AI implementation across various industries, highlighting the significant gap between high adoption rates and the low transformation of business processes. Understanding the factors contributing to this divide equips AI practitioners with insights into effective strategies for deploying AI solutions that not only enhance productivity but also drive substantial business outcomes, thereby informing their professional development and decision-making in future AI projects.' Summary='The report "The GenAI Divide: State of AI in Business 2025" illustrates the stark difference between high adoption and low transformative impact of generative AI (GenAI) technologies among organizations. Despite in

This cell obtains the number of input tokens from the response object and adds to "analysis".

In [14]:
analysis.InputTokens = response.usage.prompt_tokens
_logs.info(f"There are {analysis.InputTokens} input tokens.")

2026-04-18 11:34:40,912, 81661085.py, 2, INFO, There are 10936 input tokens.


This cell obtains the number of output tokens from the response object and adds to "analysis".

In [15]:
analysis.OutputTokens = response.usage.completion_tokens
_logs.info(f"There are {analysis.InputTokens} output tokens.")

2026-04-18 11:34:40,917, 2390898528.py, 2, INFO, There are 10936 output tokens.


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
